In [1]:
import mtl
import pandas as pd
import os

#print(os.getcwd())
#print(os.listdir())
#print(os.getcwd())
#os.chdir("c:/Users/Gugu/Documents/UAntwerpen/mosis-assignment-1")

#print(os.getcwd())

CURRENT_DIR = os.getcwd()

PROJECT_ROOT = os.path.abspath(
    os.path.join(CURRENT_DIR, "..", "..")
)

os.chdir(PROJECT_ROOT)

print(os.getcwd())

/Users/zlata/Library/CloudStorage/OneDrive-Personal/UAntwerpen/1_semester/Modelling_of_Software_Intensive_Systems/mosis_1


In [2]:
data=["data/stl/car trolley/car_system_kp100_ki1_kd100.csv",
     "data/stl/car trolley/car_system_kp210_ki1_kd1.csv",
      "data/stl/car trolley/car_system_kp250_ki5_kd5.csv",
       "data/stl/car trolley/car_system_kp280_ki10_kd10.csv",
        "data/stl/car trolley/car_system_kp300_ki1_kd20.csv",
         "data/stl/car trolley/car_system_kp300_ki20_kd1.csv",
          "data/stl/car trolley/car_system_kp320_ki15_kd5.csv",
           "data/stl/car trolley/car_system_kp350_ki20_kd10.csv",
             "data/stl/car trolley/car_system_kp370_ki10_kd15.csv",
             "data/stl/car trolley/car_system_kp390_ki5_kd20.csv"]

In [3]:
traces=[]

for file in data:
    frame = pd.read_csv(file)
    traces.append({
        #reaches_soon: within the first 20 s, the gap is within 0.5 m of 10.
        "reaches_soon": list(zip(frame["time"], (frame["intervehicular_distance.y"] >= 9.5) & (frame["intervehicular_distance.y"] <= 10.5))),
        #settled_tight: from time 30 through the end, the gap stays within 0.5 m of 10.
        "settled_tight": list(zip(frame["time"], (frame["intervehicular_distance.y"] >= 9.5) & (frame["intervehicular_distance.y"] <= 10.5))),
        #settled_loose: from time 20 through the end, the gap stays within 1 m of 10.
        "settled_loose": list(zip(frame["time"], (frame["intervehicular_distance.y"].abs()>=9.0) &  (frame["intervehicular_distance.y"].abs()<=11.0) ) ),
        #keeps_distance: the gap is at least 9 m at every sample.
        "keeps_distance": list(zip(frame["time"], (frame["intervehicular_distance.y"].abs()>=9.0) ) ),
        #accel_bounded: the absolute acceleration is at most 10 m/s² at every sample.
        "accel_bounded": list(zip(frame["time"], (frame["ego_car.accSensor.a"].abs()<=10.0) ) )


    })
reaches_soon = mtl.parse("F[0, 20.1] (reaches_soon)")
settled_tight = mtl.parse("G[30, 60.1] (settled_tight)")
settled_loose = mtl.parse("G[20,60.1 ] (settled_loose)")
keeps_distance= mtl.parse("G[0,60.1 ] (keeps_distance)")
accel_bounded = mtl.parse("G[0,60.1 ] (accel_bounded)")

properties = {
    "reaches_soon": reaches_soon,
    "settled_tight": settled_tight,
    "settled_loose": settled_loose,
    "keeps_distance": keeps_distance,
    "accel_bounded": accel_bounded

}


results = []

for i, trace in enumerate(traces):

    row = {
        "trace": os.path.basename(data[i])
    }

    for name, phi in properties.items():

        row[name] = phi( trace,time=0,quantitative=False,dt=1)

    results.append(row)
    

In [4]:
results_df = pd.DataFrame(results)

print(results_df)
results_df= results_df.replace({True: "true", False: "false"})

results_df.to_csv("results/stl/car_system_stl.csv", index=False)

                            trace  reaches_soon  settled_tight  settled_loose  \
0  car_system_kp100_ki1_kd100.csv          True           True           True   
1    car_system_kp210_ki1_kd1.csv          True          False           True   
2    car_system_kp250_ki5_kd5.csv          True           True           True   
3  car_system_kp280_ki10_kd10.csv          True           True           True   
4   car_system_kp300_ki1_kd20.csv          True           True           True   
5   car_system_kp300_ki20_kd1.csv          True          False          False   
6   car_system_kp320_ki15_kd5.csv          True           True           True   
7  car_system_kp350_ki20_kd10.csv          True           True           True   
8  car_system_kp370_ki10_kd15.csv          True           True           True   
9   car_system_kp390_ki5_kd20.csv          True           True           True   

   keeps_distance  accel_bounded  
0            True           True  
1            True           True  
2  